# Notebook for the 1000 Runs Ensemble

In [1]:
import pandas as pd
import os
from utils.eda_utils import EDAUtils
import boto3

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
SCRIPT_DIR_PATH = os.getcwd()
ROOT_DIR_PATH = os.path.dirname(SCRIPT_DIR_PATH)
DATA_DIR_PATH = os.path.join(ROOT_DIR_PATH, "data")
MAPPING_DIR_PATH = os.path.join(DATA_DIR_PATH, "mapping")
SSP_DIR_PATH = os.path.join(DATA_DIR_PATH, "ssp")
TRAINING_DIR_PATH = os.path.join(DATA_DIR_PATH, "training")
CONFIG_DIR_PATH = os.path.join(ROOT_DIR_PATH, "config")

In [4]:
os.makedirs(SSP_DIR_PATH, exist_ok=True)

In [4]:
edau = EDAUtils()

## Pull data from AWS S3


In [6]:
aws_config = edau.read_yaml(os.path.join(CONFIG_DIR_PATH, "aws_credentials_config.yaml"))
profile_name = aws_config["profile_name"]
bucket_name = aws_config["bucket_name"]
# Set your profile
session = boto3.Session(profile_name=profile_name)

# Create an S3 client or resource
s3 = session.resource('s3')

run_id = "sisepuede_run_2025-08-28t15;29;22.344855"

# Define folder prefix
prefix = f'transfers/{run_id}/'  # this is like the "folder" in S3

In [7]:
# Local destination
destination = os.path.join(SSP_DIR_PATH, run_id)
if os.path.exists(destination) and os.listdir(destination):
    print(f"Destination '{destination}' already exists and is not empty. Skipping download.")
else:
    os.makedirs(destination, exist_ok=True)
    bucket = s3.Bucket(bucket_name)
    for obj in bucket.objects.filter(Prefix=prefix):
        if obj.key.endswith('/') or "transformations" in obj.key:  # skip directories and transformations
            continue
        target_path = os.path.join(destination, os.path.basename(obj.key))
        print(f"Downloading {obj.key} to {target_path}")
        bucket.download_file(obj.key, target_path)

In [5]:
run_id="sisepuede_run_2025-08-28t15;29;22.344855"

In [6]:
SIMULATION_DIR_PATH = os.path.join(SSP_DIR_PATH, run_id)
print(SIMULATION_DIR_PATH)

e:\Current_2023\WI\work\ssp_louisiana\metamodel\data\ssp\sisepuede_run_2025-08-28t15;29;22.344855


## Load and Process LHC Samples Dataframes

In [7]:
# Load lhc samples dfs
lhs_exogenous_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "ATTRIBUTE_LHC_SAMPLES_EXOGENOUS_UNCERTAINTIES.csv"))
lhs_levers_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "ATTRIBUTE_LHC_SAMPLES_LEVER_EFFECTS.csv"))

In [8]:
# Check design ids
lhs_exogenous_df.head()

,region,design_id,future_id,47,48,49,50,51,52,53,54,55,56,57,58
0,louisiana,-1,1,0.639363,0.261333,0.115895,0.456909,0.389376,0.065254,0.452244,0.395609,0.190891,0.681412,0.199758,0.969220
1,louisiana,-1,2,0.524422,0.141191,0.625291,0.369740,0.475667,0.017979,0.912414,0.894009,0.046745,0.000798,0.725034,0.902345
2,louisiana,-1,3,0.438775,0.902956,0.099015,0.511602,0.405292,0.416929,0.832413,0.406103,0.929365,0.575067,0.208350,0.175365
3,louisiana,-1,4,0.025503,0.833553,0.666840,0.301819,0.569698,0.778987,0.995629,0.994153,0.946893,0.506764,0.227383,0.113647
4,louisiana,-1,5,0.225323,0.718339,0.291168,0.507558,0.499419,0.918567,0.183357,0.397512,0.328322,0.362234,0.947292,0.764492


In [9]:
lhs_exogenous_df.design_id.unique()

array([-1,  4])

In [10]:
lhs_levers_df.head()

,region,design_id,future_id,1,2,3,4,5,6,7,...,1753,1754,1759,1760,1762,1770,1772,1784,1785,1793
0,louisiana,-1,1,0.354682,0.681300,0.670551,0.371381,0.597378,0.390717,0.554668,...,0.406492,0.785543,0.249585,0.808708,0.824054,0.596933,0.085905,0.349450,0.718493,0.920739
1,louisiana,-1,2,0.023662,0.659807,0.066549,0.350365,0.536160,0.040042,0.101776,...,0.133208,0.725231,0.615082,0.234993,0.342072,0.975031,0.431114,0.700254,0.246668,0.536234
2,louisiana,-1,3,0.414641,0.092022,0.227137,0.020454,0.832906,0.293800,0.601890,...,0.320059,0.836571,0.386288,0.728487,0.069142,0.128675,0.699301,0.166701,0.354644,0.300748
3,louisiana,-1,4,0.307337,0.204967,0.050181,0.805452,0.558369,0.506111,0.974177,...,0.318834,0.118582,0.767219,0.762363,0.120502,0.838756,0.298101,0.885792,0.274482,0.827713
4,louisiana,-1,5,0.800355,0.458285,0.101171,0.738594,0.254548,0.034029,0.673997,...,0.954028,0.331731,0.708760,0.200289,0.889109,0.238305,0.473463,0.147280,0.491020,0.480479


In [11]:
lhs_levers_df.design_id.unique()

array([-1,  4])

In [12]:
# print shapes
print(lhs_exogenous_df.shape)
print(lhs_levers_df.shape)

(2000, 15)
(2000, 73)


In [13]:
lhs_df_merged = pd.merge(lhs_exogenous_df, lhs_levers_df, on=["region", "design_id", "future_id"], how="outer", suffixes=('_X', '_L'))
lhs_df_merged.head()

,region,design_id,future_id,47,48,49,50,51,52,53,...,1753,1754,1759,1760,1762,1770,1772,1784,1785,1793
0,louisiana,-1,1,0.639363,0.261333,0.115895,0.456909,0.389376,0.065254,0.452244,...,0.406492,0.785543,0.249585,0.808708,0.824054,0.596933,0.085905,0.349450,0.718493,0.920739
1,louisiana,-1,2,0.524422,0.141191,0.625291,0.369740,0.475667,0.017979,0.912414,...,0.133208,0.725231,0.615082,0.234993,0.342072,0.975031,0.431114,0.700254,0.246668,0.536234
2,louisiana,-1,3,0.438775,0.902956,0.099015,0.511602,0.405292,0.416929,0.832413,...,0.320059,0.836571,0.386288,0.728487,0.069142,0.128675,0.699301,0.166701,0.354644,0.300748
3,louisiana,-1,4,0.025503,0.833553,0.666840,0.301819,0.569698,0.778987,0.995629,...,0.318834,0.118582,0.767219,0.762363,0.120502,0.838756,0.298101,0.885792,0.274482,0.827713
4,louisiana,-1,5,0.225323,0.718339,0.291168,0.507558,0.499419,0.918567,0.183357,...,0.954028,0.331731,0.708760,0.200289,0.889109,0.238305,0.473463,0.147280,0.491020,0.480479


In [14]:
# Filter the lhs_df_merged to only include rows where design_id is 4
lhs_df_merged = lhs_df_merged[lhs_df_merged.design_id == 4]
lhs_df_merged.shape

(1000, 85)

In [15]:
lhs_df_merged.design_id.unique()

array([4])

In [16]:
# NOTE: check col names, there should be no duplicates
lhs_df_merged.columns

Index(['region', 'design_id', 'future_id', '47', '48', '49', '50', '51', '52',
       '53', '54', '55', '56', '57', '58', '1', '2', '3', '4', '5', '6', '7',
       '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19',
       '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31',
       '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43',
       '44', '45', '46', '1482', '1483', '1485', '1487', '1706', '1707',
       '1710', '1712', '1715', '1732', '1733', '1736', '1739', '1741', '1753',
       '1754', '1759', '1760', '1762', '1770', '1772', '1784', '1785', '1793'],
      dtype='object')

In [17]:
lhs_df_merged.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1000 entries, 1000 to 1999
Data columns (total 85 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   region     1000 non-null   object 
 1   design_id  1000 non-null   int64  
 2   future_id  1000 non-null   int64  
 3   47         1000 non-null   float64
 4   48         1000 non-null   float64
 5   49         1000 non-null   float64
 6   50         1000 non-null   float64
 7   51         1000 non-null   float64
 8   52         1000 non-null   float64
 9   53         1000 non-null   float64
 10  54         1000 non-null   float64
 11  55         1000 non-null   float64
 12  56         1000 non-null   float64
 13  57         1000 non-null   float64
 14  58         1000 non-null   float64
 15  1          1000 non-null   float64
 16  2          1000 non-null   float64
 17  3          1000 non-null   float64
 18  4          1000 non-null   float64
 19  5          1000 non-null   float64
 20  6         

## Load SISEPUEDE WIDE_INPUTS_OUTPUTS

In [18]:
wide_inputs_outputs_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "sisepuede_results_IDE_6004_filtered.csv"))
wide_inputs_outputs_df

,primary_id,region,time_period,area_agrc_crops_bevs_and_spices,area_agrc_crops_cereals,area_agrc_crops_fibers,area_agrc_crops_fruits,area_agrc_crops_herbs_and_other_perennial_crops,area_agrc_crops_nuts,area_agrc_crops_other_annual,...,emission_co2e_subsector_total_inen,emission_co2e_subsector_total_ippu,emission_co2e_subsector_total_lndu,emission_co2e_subsector_total_lsmm,emission_co2e_subsector_total_lvst,emission_co2e_subsector_total_scoe,emission_co2e_subsector_total_soil,emission_co2e_subsector_total_trns,emission_co2e_subsector_total_trww,emission_co2e_subsector_total_waso
0,332332,louisiana,7,0,356696.043492,66146.621297,77.773211,76769.151304,6508.599365,1.119173e+06,...,117.042622,2.762559,-0.077001,0.230319,1.539811,4.491483,1.233181,45.130223,0.447204,3.138402
1,332332,louisiana,8,0,355221.860914,65873.245131,77.451784,76451.873477,6481.700093,1.114548e+06,...,160.204394,2.773548,-0.104854,0.229389,1.516623,4.525427,1.226921,45.865290,0.454013,3.189568
2,332332,louisiana,9,0,353750.075712,65600.313541,77.130879,76135.111621,6454.844566,1.109930e+06,...,116.085554,2.786645,-0.132605,0.228545,1.493794,4.561015,1.210181,46.698636,0.461144,3.239706
3,332332,louisiana,10,0,352280.764654,65327.840762,76.810514,75818.882257,6428.034184,1.105320e+06,...,163.932900,2.801782,-0.160253,0.227765,1.471320,4.598400,1.181498,47.611029,0.468528,3.291336
4,332332,louisiana,11,0,350814.002751,65055.840705,76.490704,75503.201530,6401.270317,1.100718e+06,...,163.431885,2.818877,-0.187795,0.227044,1.449198,4.637649,1.140642,48.587463,0.476114,3.343789
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28763,403402,louisiana,31,0,362765.501037,62973.310285,84.270736,82487.288241,7362.303302,1.185080e+06,...,169.371247,2.075427,-1.990093,0.193981,1.441077,1.211796,1.706858,36.976660,0.504737,5.271994
28764,403402,louisiana,32,0,365228.715699,63193.456636,85.153817,83260.287378,7453.568466,1.195508e+06,...,169.256140,2.031856,-2.010800,0.192032,1.456824,1.041882,1.778988,36.260617,0.504923,5.373060
28765,403402,louisiana,33,0,367719.988263,63434.430550,86.031953,84023.290062,7543.331453,1.205869e+06,...,169.154549,1.986896,-2.048092,0.189841,1.471911,0.871422,1.846095,35.524647,0.504966,5.474498
28766,403402,louisiana,34,0,370220.245668,63693.168311,86.899932,84771.705654,7631.119506,1.216095e+06,...,169.064989,1.940356,-2.098197,0.187394,1.486149,0.700168,1.891870,34.765612,0.504860,5.576305


In [19]:
wide_inputs_outputs_df.primary_id.nunique()

992

## Load Costs-Benefits Data

In [20]:
cb_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "wide_cb_data_lhc_2025-08-28t15;29;22.344855.csv"))
cb_df.head()

,primary_id,future_id,strategy_code,Year,air_pollution,congestion,consumer_savings,crop_value,ecosystem_services,env_pollution,...,human_health,ippu_value,land_pollution,lvst_value,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution
0,402402,0,PFLO:ALL_LA_ACTIONS,2022.0,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.0,...,0.000000,0.0,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.0
1,402402,0,PFLO:ALL_LA_ACTIONS,2023.0,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.0,...,0.000000,0.0,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.0
2,402402,0,PFLO:ALL_LA_ACTIONS,2024.0,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.0,...,0.000000,0.0,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.0
3,402402,0,PFLO:ALL_LA_ACTIONS,2025.0,9.695006e-16,-1.451481e-23,0.021749,0.000000e+00,6.959355e-12,0.0,...,0.019772,0.0,0.000000e+00,0.0,-2.494771e-23,5.093170e-14,0.000000e+00,-0.021329,0.000000e+00,0.0
4,402402,0,PFLO:ALL_LA_ACTIONS,2026.0,1.425974e-16,0.000000e+00,0.043782,2.673078e-14,7.691665e-12,0.0,...,0.039802,0.0,-5.968559e-17,0.0,0.000000e+00,-5.563550e-14,-5.269051e-15,-0.072557,8.114148e-17,0.0


In [21]:
cb_df.future_id.nunique()

991

In [22]:
cb_df.primary_id.nunique()

991

## Data Cleaning

### SISEPUEDE Emission data

In [23]:
# Get the subsector total variables
subsector_total_vars = [c for c in wide_inputs_outputs_df.columns if "emission_co2e_subsector_total" in c]

In [24]:
# Filter to only subsector total columns and primary_id, time_period
la_emissions_df = wide_inputs_outputs_df[["primary_id", "time_period"] + subsector_total_vars]
la_emissions_df.head()

,primary_id,time_period,emission_co2e_subsector_total_agrc,emission_co2e_subsector_total_ccsq,emission_co2e_subsector_total_entc,emission_co2e_subsector_total_fgtv,emission_co2e_subsector_total_frst,emission_co2e_subsector_total_inen,emission_co2e_subsector_total_ippu,emission_co2e_subsector_total_lndu,emission_co2e_subsector_total_lsmm,emission_co2e_subsector_total_lvst,emission_co2e_subsector_total_scoe,emission_co2e_subsector_total_soil,emission_co2e_subsector_total_trns,emission_co2e_subsector_total_trww,emission_co2e_subsector_total_waso
0,332332,7,2.924249,0.0,35.431379,13.094104,-36.545088,117.042622,2.762559,-0.077001,0.230319,1.539811,4.491483,1.233181,45.130223,0.447204,3.138402
1,332332,8,2.864242,0.0,41.700805,14.473550,-39.679345,160.204394,2.773548,-0.104854,0.229389,1.516623,4.525427,1.226921,45.865290,0.454013,3.189568
2,332332,9,2.912181,0.0,40.757204,14.360506,-42.096028,116.085554,2.786645,-0.132605,0.228545,1.493794,4.561015,1.210181,46.698636,0.461144,3.239706
3,332332,10,2.900085,0.0,50.543233,15.596323,-44.041357,163.932900,2.801782,-0.160253,0.227765,1.471320,4.598400,1.181498,47.611029,0.468528,3.291336
4,332332,11,2.888010,0.0,50.613036,15.796512,-45.670588,163.431885,2.818877,-0.187795,0.227044,1.449198,4.637649,1.140642,48.587463,0.476114,3.343789


In [25]:
la_emissions_df.tail()

,primary_id,time_period,emission_co2e_subsector_total_agrc,emission_co2e_subsector_total_ccsq,emission_co2e_subsector_total_entc,emission_co2e_subsector_total_fgtv,emission_co2e_subsector_total_frst,emission_co2e_subsector_total_inen,emission_co2e_subsector_total_ippu,emission_co2e_subsector_total_lndu,emission_co2e_subsector_total_lsmm,emission_co2e_subsector_total_lvst,emission_co2e_subsector_total_scoe,emission_co2e_subsector_total_soil,emission_co2e_subsector_total_trns,emission_co2e_subsector_total_trww,emission_co2e_subsector_total_waso
28763,403402,31,2.785246,-6.424571,7.296782,10.723168,-65.661634,169.371247,2.075427,-1.990093,0.193981,1.441077,1.211796,1.706858,36.976660,0.504737,5.271994
28764,403402,32,2.791457,-6.762707,7.369101,10.312912,-66.659837,169.256140,2.031856,-2.010800,0.192032,1.456824,1.041882,1.778988,36.260617,0.504923,5.373060
28765,403402,33,2.797542,-7.100842,7.443474,9.899790,-67.669881,169.154549,1.986896,-2.048092,0.189841,1.471911,0.871422,1.846095,35.524647,0.504966,5.474498
28766,403402,34,2.803377,-7.438977,7.519861,9.483770,-68.689916,169.064989,1.940356,-2.098197,0.187394,1.486149,0.700168,1.891870,34.765612,0.504860,5.576305
28767,403402,35,2.808861,-7.777113,7.597122,9.063926,-69.719472,169.136966,1.892021,-2.123466,0.184680,1.499386,0.527891,1.923271,33.979674,0.504593,5.678443


### Production Data

In [26]:
# Get the subsector total variables
industry_value_fuel_vars = [c for c in wide_inputs_outputs_df.columns if "totalvalue_enfu_fuel_consumed_inen" in c]

In [27]:
# Filter to only production columns avoiding "subsector" total columns
industrial_production_df = wide_inputs_outputs_df[["primary_id", "time_period"] + industry_value_fuel_vars]
industrial_production_df

,primary_id,time_period,totalvalue_enfu_fuel_consumed_inen_fuel_biomass,totalvalue_enfu_fuel_consumed_inen_fuel_coal,totalvalue_enfu_fuel_consumed_inen_fuel_coke,totalvalue_enfu_fuel_consumed_inen_fuel_diesel,totalvalue_enfu_fuel_consumed_inen_fuel_electricity,totalvalue_enfu_fuel_consumed_inen_fuel_furnace_gas,totalvalue_enfu_fuel_consumed_inen_fuel_gasoline,totalvalue_enfu_fuel_consumed_inen_fuel_hydrocarbon_gas_liquids,totalvalue_enfu_fuel_consumed_inen_fuel_hydrogen,totalvalue_enfu_fuel_consumed_inen_fuel_kerosene,totalvalue_enfu_fuel_consumed_inen_fuel_natural_gas,totalvalue_enfu_fuel_consumed_inen_fuel_oil
0,332332,7,3.656308,0.406256,178.382747,1170.618595,5.247586e+07,181.381237,0,0.323105,0,1989.574882,276.643486,10.053944
1,332332,8,0.577292,0.064144,182.041290,1138.090449,5.212249e+07,174.739848,0,0.353500,0,1743.288173,289.544575,10.129708
2,332332,9,0.232957,0.025884,194.289929,1125.644561,5.101550e+07,180.160326,0,0.371168,0,1697.631288,309.488469,10.209134
3,332332,10,0.192548,0.021394,194.550217,1099.683741,4.965545e+07,179.308108,0,0.395631,0,1721.287445,308.523220,10.292580
4,332332,11,0.150090,0.016677,194.299593,1087.864099,4.823023e+07,176.765950,0,0.416305,0,1749.876854,302.778959,10.380384
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28763,403402,31,0.000000,0.000000,227.569112,889.805318,2.366770e+07,223.095027,0,17.698829,0,2137.856307,484.051304,2.696082
28764,403402,32,0.000000,0.000000,228.635240,886.574213,2.310498e+07,224.198384,0,17.703967,0,2155.618805,488.838082,2.317989
28765,403402,33,0.000000,0.000000,229.016041,883.751350,2.258153e+07,225.301739,0,17.654904,0,2179.820574,492.279820,1.938658
28766,403402,34,0.000000,0.000000,230.204322,876.459039,2.209217e+07,225.335722,0,17.553265,0,2199.368038,496.540828,1.557596


In [28]:
# 2) Define your fuels and sectors
relevant_fuels = ['biomass', 
                    'coal', 
                    'coke', 
                    'diesel', 
                    'electricity',
                    'furnace_gas',
                    'gasoline', 
                    'hydrocarbon_gas_liquids',
                    'hydrogen',
                    'kerosene',
                    'natural_gas',
                    'oil']

sectors = ['agriculture_and_livestock',
           'cement',
           'chemicals',
           'electronics',
           'glass',
           'lime_and_carbonite',
           'metals',
           'mining',
           'other_product_manufacturing',
           'paper',
           'plastic',
           'recycled_glass',
           'recycled_metals',
           'recycled_paper',
           'recycled_plastic',
           'recycled_rubber_and_leather',
           'recycled_textiles',
           'recycled_wood',
           'rubber_and_leather',
           'textiles',
           'wood']

In [29]:
# 4) Industrial cost parameters
capex_industrial_electricity = 92666.6 * 21
capex_industrial_other       = 92666.6 * 12
opex_industrial_electricity  = 92666.6 * 2.5
opex_industrial_other        = 92666.6 * 4.5
capex_multiplier_efficiency = 10000000
opex_multiplier_efficiency = 0

In [30]:
# 5) Loop over fuels and sectors
# Use the original dataframe directly
# Initialize results dataframe
ind_fuel_demand_by_sector = pd.DataFrame({
    'primary_id': wide_inputs_outputs_df['primary_id'],
    'time_period': wide_inputs_outputs_df['time_period']
}, index=wide_inputs_outputs_df.index)

# Loop over fuels and sectors
for fuel in relevant_fuels:
    # efficiency column for this fuel
    eff_cols = [c for c in wide_inputs_outputs_df.columns
                if c.startswith(f'efficfactor_enfu_industrial_energy_fuel_{fuel}')]
    if not eff_cols:
        continue
    fuel_efficiency = wide_inputs_outputs_df[eff_cols[0]]

    for sector in sectors:
        sector_dem_cols = [c for c in wide_inputs_outputs_df.columns
                           if f'energy_demand_inen_{sector}' in c]
        sector_fuel_fraction_cols = [c for c in wide_inputs_outputs_df.columns
                                     if f'frac_inen_energy_{sector}_{fuel}' in c]

        if sector_dem_cols and sector_fuel_fraction_cols:
            sector_total_demand = wide_inputs_outputs_df[sector_dem_cols[0]]
            sector_fuel_fraction = wide_inputs_outputs_df[sector_fuel_fraction_cols[0]]

            if (sector_fuel_fraction * sector_total_demand).sum() > 0 or fuel == 'electricity':
                sector_fuel_demand = sector_fuel_fraction * sector_total_demand
                ind_fuel_demand_by_sector[f'energy_demand_{sector}_{fuel}'] = sector_fuel_demand

                # CAPEX/OPEX
                if fuel == 'electricity':
                    ind_fuel_demand_by_sector[f'energy_demand_capex_{sector}_{fuel}'] = (
                        sector_fuel_demand * capex_industrial_electricity
                    )
                    ind_fuel_demand_by_sector[f'energy_demand_opex_{sector}_{fuel}'] = (
                        sector_fuel_demand * opex_industrial_electricity
                    )
                else:
                    ind_fuel_demand_by_sector[f'energy_demand_capex_{sector}_{fuel}'] = (
                        sector_fuel_demand * capex_industrial_other
                    )
                    ind_fuel_demand_by_sector[f'energy_demand_opex_{sector}_{fuel}'] = (
                        sector_fuel_demand * opex_industrial_other
                    )

                # Fuel consumed
                sector_fuel_consumed = sector_fuel_demand / fuel_efficiency

                # Baseline = first time_period per primary_id
                sector_fuel_consumed_baseline = (
                    sector_fuel_consumed.groupby(wide_inputs_outputs_df['primary_id'])
                                        .transform('first')
                )

                # Change relative to baseline
                sector_change_in_fuel_consumed = (
                    sector_fuel_consumed_baseline - sector_fuel_consumed
                )

                # Save results
                ind_fuel_demand_by_sector[f'efficiency_energy_saving_{sector}_{fuel}'] = (
                    sector_change_in_fuel_consumed
                )
                ind_fuel_demand_by_sector[f'efficiency_capex_{sector}_{fuel}'] = (
                    sector_change_in_fuel_consumed * capex_multiplier_efficiency
                )
                ind_fuel_demand_by_sector[f'efficiency_opex_{sector}_{fuel}'] = (
                    sector_change_in_fuel_consumed * opex_multiplier_efficiency
                )


C:\Users\nasta\AppData\Local\Temp\ipykernel_45748\3509172589.py:63: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ind_fuel_demand_by_sector[f'efficiency_energy_saving_{sector}_{fuel}'] = (
C:\Users\nasta\AppData\Local\Temp\ipykernel_45748\3509172589.py:66: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ind_fuel_demand_by_sector[f'efficiency_capex_{sector}_{fuel}'] = (
C:\Users\nasta\AppData\Local\Temp\ipykernel_45748\3509172589.py:69: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `fram

In [33]:
ind_fuel_demand_by_sector

,primary_id,time_period,energy_demand_cement_coal,energy_demand_capex_cement_coal,energy_demand_opex_cement_coal,efficiency_energy_saving_cement_coal,efficiency_capex_cement_coal,efficiency_opex_cement_coal,energy_demand_chemicals_coal,energy_demand_capex_chemicals_coal,...,energy_demand_opex_other_product_manufacturing_oil,efficiency_energy_saving_other_product_manufacturing_oil,efficiency_capex_other_product_manufacturing_oil,efficiency_opex_other_product_manufacturing_oil,energy_demand_recycled_wood_oil,energy_demand_capex_recycled_wood_oil,energy_demand_opex_recycled_wood_oil,efficiency_energy_saving_recycled_wood_oil,efficiency_capex_recycled_wood_oil,efficiency_opex_recycled_wood_oil
0,332332,7,15.110409,1.680276e+07,6.301036e+06,0.000000,0.000000e+00,0.0,0.000732,813.906668,...,1.065403e+07,0.000000,0.000000e+00,0.0,0.319755,355567.121778,133337.670667,0.000000,0.000000e+00,0.0
1,332332,8,12.290454,1.366697e+07,5.125115e+06,4.762860,4.762860e+07,0.0,0.000070,77.506444,...,1.101825e+07,-1.019247,-1.019247e+07,-0.0,0.314100,349279.179225,130979.692209,0.009181,9.181124e+04,0.0
2,332332,9,11.835264,1.316080e+07,4.935302e+06,5.593867,5.593867e+07,0.0,0.000000,0.000000,...,1.138607e+07,-2.041192,-2.041192e+07,-0.0,0.306088,340369.779635,127638.667363,0.021401,2.140106e+05,0.0
3,332332,10,11.597945,1.289691e+07,4.836340e+06,6.061209,6.061209e+07,0.0,0.000000,0.000000,...,1.179935e+07,-3.197407,-3.197407e+07,-0.0,0.297894,331257.757247,124221.658968,0.033769,3.376881e+05,0.0
4,332332,11,10.757587,1.196243e+07,4.485911e+06,7.510201,7.510201e+07,0.0,0.000000,0.000000,...,1.222929e+07,-4.395974,-4.395974e+07,-0.0,0.290604,323151.473668,121181.802625,0.044862,4.486193e+05,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28763,403402,31,0.000000,0.000000e+00,0.000000e+00,25.063958,2.506396e+08,0.0,0.000000,0.000000,...,1.480369e+07,-5.738279,-5.738279e+07,-0.0,0.089217,99208.904293,37203.339110,0.326814,3.268142e+06,0.0
28764,403402,32,0.000000,0.000000e+00,0.000000e+00,25.063958,2.506396e+08,0.0,0.000000,0.000000,...,1.488998e+07,-5.700835,-5.700835e+07,-0.0,0.083299,92628.244117,34735.591544,0.334052,3.340518e+06,0.0
28765,403402,33,0.000000,0.000000e+00,0.000000e+00,25.063958,2.506396e+08,0.0,0.000000,0.000000,...,1.498183e+07,-5.681393,-5.681393e+07,-0.0,0.077650,86346.664049,32379.999018,0.340890,3.408901e+06,0.0
28766,403402,34,0.000000,0.000000e+00,0.000000e+00,25.063958,2.506396e+08,0.0,0.000000,0.000000,...,1.507977e+07,-5.680952,-5.680952e+07,-0.0,0.072252,80343.859049,30128.947143,0.347358,3.473585e+06,0.0


In [31]:
ind_fuel_demand_by_sector.primary_id.nunique()

992

In [32]:
ind_fuel_demand_by_sector.isna().sum().sum() 

np.int64(0)

### CB Data

In [33]:
# Make all column names lowercase
cb_df.columns = [c.lower() for c in cb_df.columns]

# Filter to only important cb columns
cb_df = cb_df[["primary_id",
               "future_id",
               "year",
               "technical_cost",
               "consumer_savings",
               #"human_health",
               "air_pollution"]]

cb_df.head()

,primary_id,future_id,year,technical_cost,consumer_savings,air_pollution
0,402402,0,2022.0,0.000000,0.000000,0.000000e+00
1,402402,0,2023.0,0.000000,0.000000,0.000000e+00
2,402402,0,2024.0,0.000000,0.000000,0.000000e+00
3,402402,0,2025.0,-0.021329,0.021749,9.695006e-16
4,402402,0,2026.0,-0.072557,0.043782,1.425974e-16


In [34]:
cb_df.tail()

,primary_id,future_id,year,technical_cost,consumer_savings,air_pollution
28734,403402,1000,2046.0,2.793969,0.549269,3.885243
28735,403402,1000,2047.0,1.528156,0.576659,4.035926
28736,403402,1000,2048.0,9.246420,0.604128,4.186431
28737,403402,1000,2049.0,1.451072,0.631667,4.340256
28738,403402,1000,2050.0,1.090039,0.659271,4.500326


In [35]:
cb_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28739 entries, 0 to 28738
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   primary_id        28739 non-null  int64  
 1   future_id         28739 non-null  int64  
 2   year              28739 non-null  float64
 3   technical_cost    28739 non-null  float64
 4   consumer_savings  28739 non-null  float64
 5   air_pollution     28739 non-null  float64
dtypes: float64(4), int64(2)
memory usage: 1.3 MB


## LHS Data

In [36]:
lhs_df_merged = lhs_df_merged.drop(columns=["design_id", "region"])
lhs_df_merged.head()

,future_id,47,48,49,50,51,52,53,54,55,...,1753,1754,1759,1760,1762,1770,1772,1784,1785,1793
1000,1,0.639363,0.261333,0.115895,0.456909,0.389376,0.065254,0.452244,0.395609,0.190891,...,0.406492,0.785543,0.249585,0.808708,0.824054,0.596933,0.085905,0.349450,0.718493,0.920739
1001,2,0.524422,0.141191,0.625291,0.369740,0.475667,0.017979,0.912414,0.894009,0.046745,...,0.133208,0.725231,0.615082,0.234993,0.342072,0.975031,0.431114,0.700254,0.246668,0.536234
1002,3,0.438775,0.902956,0.099015,0.511602,0.405292,0.416929,0.832413,0.406103,0.929365,...,0.320059,0.836571,0.386288,0.728487,0.069142,0.128675,0.699301,0.166701,0.354644,0.300748
1003,4,0.025503,0.833553,0.666840,0.301819,0.569698,0.778987,0.995629,0.994153,0.946893,...,0.318834,0.118582,0.767219,0.762363,0.120502,0.838756,0.298101,0.885792,0.274482,0.827713
1004,5,0.225323,0.718339,0.291168,0.507558,0.499419,0.918567,0.183357,0.397512,0.328322,...,0.954028,0.331731,0.708760,0.200289,0.889109,0.238305,0.473463,0.147280,0.491020,0.480479


## Transform time series format into single-row format

### SISEPUEDE Emission data

In [37]:
# Sum all the subsector emission columns across axis=1
la_emission_total_df = la_emissions_df.copy()
la_emission_total_df["emission_total"] = la_emission_total_df[subsector_total_vars].sum(axis=1)
la_emission_total_df.head()

,primary_id,time_period,emission_co2e_subsector_total_agrc,emission_co2e_subsector_total_ccsq,emission_co2e_subsector_total_entc,emission_co2e_subsector_total_fgtv,emission_co2e_subsector_total_frst,emission_co2e_subsector_total_inen,emission_co2e_subsector_total_ippu,emission_co2e_subsector_total_lndu,emission_co2e_subsector_total_lsmm,emission_co2e_subsector_total_lvst,emission_co2e_subsector_total_scoe,emission_co2e_subsector_total_soil,emission_co2e_subsector_total_trns,emission_co2e_subsector_total_trww,emission_co2e_subsector_total_waso,emission_total
0,332332,7,2.924249,0.0,35.431379,13.094104,-36.545088,117.042622,2.762559,-0.077001,0.230319,1.539811,4.491483,1.233181,45.130223,0.447204,3.138402,190.843446
1,332332,8,2.864242,0.0,41.700805,14.473550,-39.679345,160.204394,2.773548,-0.104854,0.229389,1.516623,4.525427,1.226921,45.865290,0.454013,3.189568,239.239570
2,332332,9,2.912181,0.0,40.757204,14.360506,-42.096028,116.085554,2.786645,-0.132605,0.228545,1.493794,4.561015,1.210181,46.698636,0.461144,3.239706,192.566476
3,332332,10,2.900085,0.0,50.543233,15.596323,-44.041357,163.932900,2.801782,-0.160253,0.227765,1.471320,4.598400,1.181498,47.611029,0.468528,3.291336,250.422588
4,332332,11,2.888010,0.0,50.613036,15.796512,-45.670588,163.431885,2.818877,-0.187795,0.227044,1.449198,4.637649,1.140642,48.587463,0.476114,3.343789,249.551836


In [38]:
# Keep only the primary_id, time_period, and emission_total columns
la_emission_total_df = la_emission_total_df[["primary_id", "time_period", "emission_total"]]
la_emission_total_df.head()

,primary_id,time_period,emission_total
0,332332,7,190.843446
1,332332,8,239.239570
2,332332,9,192.566476
3,332332,10,250.422588
4,332332,11,249.551836


In [39]:
la_emission_total_df.tail()

,primary_id,time_period,emission_total
28763,403402,31,165.482674
28764,403402,32,162.936447
28765,403402,33,160.346816
28766,403402,34,157.697621
28767,403402,35,155.176785


### Emission data sum

In [40]:
# aggregate data by primary_id summing the emissions
la_emission_df_sum_agg = la_emission_total_df.groupby(["primary_id"]).sum().reset_index()

# Drop time period column
la_emission_df_sum_agg = la_emission_df_sum_agg.drop(columns=["time_period"])
la_emission_df_sum_agg.head()

,primary_id,emission_total
0,332332,8387.685774
1,402402,4717.592913
2,402403,6648.349151
3,402404,7129.331813
4,402405,6396.992675


### Emission data mean

In [41]:
# Filter out rows with time_period < 31
la_filtered_emission_total_df = la_emission_total_df[la_emission_total_df["time_period"] >= 31]
la_filtered_emission_total_df = la_filtered_emission_total_df.reset_index(drop=True)
la_filtered_emission_total_df.head(7)

,primary_id,time_period,emission_total
0,332332,31,320.440810
1,332332,32,322.349545
2,332332,33,324.392922
3,332332,34,326.574698
4,332332,35,329.135650
5,402402,31,74.619462
6,402402,32,68.820484


In [42]:
# aggregate data by primary_id by summing the emissions
la_emission_df_mean_agg = la_filtered_emission_total_df.groupby(["primary_id"]).mean().reset_index()

# Rename emission_total to emission_avg_last_five_years
la_emission_df_mean_agg.rename(columns={"emission_total": "emission_avg_last_five_years"}, inplace=True)
la_emission_df_mean_agg

,primary_id,time_period,emission_avg_last_five_years
0,332332,33.0,324.578725
1,402402,33.0,63.023067
2,402403,33.0,198.512688
3,402404,33.0,228.001674
4,402405,33.0,182.853198
...,...,...,...
987,403398,33.0,241.127083
988,403399,33.0,237.380736
989,403400,33.0,146.886074
990,403401,33.0,155.859387


In [43]:
# Drop year column as it is no longer needed
la_emission_df_mean_agg = la_emission_df_mean_agg.drop(columns=["time_period"])
la_emission_df_mean_agg.head()

,primary_id,emission_avg_last_five_years
0,332332,324.578725
1,402402,63.023067
2,402403,198.512688
3,402404,228.001674
4,402405,182.853198


### Combining emission agg into one df

In [44]:
print("la_emission_df_mean_agg shape:", la_emission_df_mean_agg.shape)
print("la_emission_df_sum_agg shape:", la_emission_df_sum_agg.shape)

la_emission_df_mean_agg shape: (992, 2)
la_emission_df_sum_agg shape: (992, 2)


In [45]:
la_emissions_df_merged = la_emission_df_mean_agg.merge(la_emission_df_sum_agg, on="primary_id", how="inner")
la_emissions_df_merged.head()

,primary_id,emission_avg_last_five_years,emission_total
0,332332,324.578725,8387.685774
1,402402,63.023067,4717.592913
2,402403,198.512688,6648.349151
3,402404,228.001674,7129.331813
4,402405,182.853198,6396.992675


In [46]:
print("la_emission_df_mean_agg shape:", la_emission_df_mean_agg.shape)

la_emission_df_mean_agg shape: (992, 2)


### Production Data

In [47]:
# Sum all the subsector emission columns across axis=1
la_production_total_df = industrial_production_df.copy()
la_production_total_df["production_total"] = la_production_total_df[industry_value_fuel_vars].sum(axis=1)
la_production_total_df.head()

,primary_id,time_period,totalvalue_enfu_fuel_consumed_inen_fuel_biomass,totalvalue_enfu_fuel_consumed_inen_fuel_coal,totalvalue_enfu_fuel_consumed_inen_fuel_coke,totalvalue_enfu_fuel_consumed_inen_fuel_diesel,totalvalue_enfu_fuel_consumed_inen_fuel_electricity,totalvalue_enfu_fuel_consumed_inen_fuel_furnace_gas,totalvalue_enfu_fuel_consumed_inen_fuel_gasoline,totalvalue_enfu_fuel_consumed_inen_fuel_hydrocarbon_gas_liquids,totalvalue_enfu_fuel_consumed_inen_fuel_hydrogen,totalvalue_enfu_fuel_consumed_inen_fuel_kerosene,totalvalue_enfu_fuel_consumed_inen_fuel_natural_gas,totalvalue_enfu_fuel_consumed_inen_fuel_oil,production_total
0,332332,7,3.656308,0.406256,178.382747,1170.618595,5.247586e+07,181.381237,0,0.323105,0,1989.574882,276.643486,10.053944,5.247967e+07
1,332332,8,0.577292,0.064144,182.041290,1138.090449,5.212249e+07,174.739848,0,0.353500,0,1743.288173,289.544575,10.129708,5.212603e+07
2,332332,9,0.232957,0.025884,194.289929,1125.644561,5.101550e+07,180.160326,0,0.371168,0,1697.631288,309.488469,10.209134,5.101902e+07
3,332332,10,0.192548,0.021394,194.550217,1099.683741,4.965545e+07,179.308108,0,0.395631,0,1721.287445,308.523220,10.292580,4.965897e+07
4,332332,11,0.150090,0.016677,194.299593,1087.864099,4.823023e+07,176.765950,0,0.416305,0,1749.876854,302.778959,10.380384,4.823375e+07


In [48]:
# Keep only the primary_id, time_period, and emission_total columns
la_production_total_df = la_production_total_df[["primary_id", "time_period", "production_total"]]
la_production_total_df.head()

,primary_id,time_period,production_total
0,332332,7,5.247967e+07
1,332332,8,5.212603e+07
2,332332,9,5.101902e+07
3,332332,10,4.965897e+07
4,332332,11,4.823375e+07


In [49]:
# aggregate data by primary_id summing the emissions
la_production_df_sum_agg = la_production_total_df.groupby(["primary_id"]).sum().reset_index()

# Drop time period column
la_production_df_sum_agg = la_production_df_sum_agg.drop(columns=["time_period"])
la_production_df_sum_agg.head()

,primary_id,production_total
0,332332,1.216840e+09
1,402402,6.787144e+08
2,402403,9.944817e+08
3,402404,7.733990e+08
4,402405,7.860210e+08


In [50]:
la_production_df_sum_agg.shape

(992, 2)

### Production Cost Data

In [51]:
industry_cost_vars = [
    c for c in ind_fuel_demand_by_sector.columns
    if c.startswith("energy_demand_capex_") or c.startswith("energy_demand_opex_")
]

In [52]:
ind_fuel_demand_by_sector[industry_cost_vars]

,energy_demand_capex_cement_coal,energy_demand_opex_cement_coal,energy_demand_capex_chemicals_coal,energy_demand_opex_chemicals_coal,energy_demand_capex_metals_coal,energy_demand_opex_metals_coal,energy_demand_capex_mining_coal,energy_demand_opex_mining_coal,energy_demand_capex_other_product_manufacturing_coal,energy_demand_opex_other_product_manufacturing_coal,...,energy_demand_capex_electronics_oil,energy_demand_opex_electronics_oil,energy_demand_capex_metals_oil,energy_demand_opex_metals_oil,energy_demand_capex_mining_oil,energy_demand_opex_mining_oil,energy_demand_capex_other_product_manufacturing_oil,energy_demand_opex_other_product_manufacturing_oil,energy_demand_capex_recycled_wood_oil,energy_demand_opex_recycled_wood_oil
0,1.680276e+07,6.301036e+06,813.906668,305.215000,829934.481644,311225.430616,174.208320,65.328120,3.593569e+06,1.347588e+06,...,1694.069129,635.275924,14470.790853,5426.546570,12954.468460,4857.925672,2.841075e+07,1.065403e+07,355567.121778,133337.670667
1,1.366697e+07,5.125115e+06,77.506444,29.064916,818590.290290,306971.358859,139.114963,52.168111,3.312895e+06,1.242336e+06,...,1790.850816,671.569056,8388.932345,3145.849629,12061.561422,4523.085533,2.938199e+07,1.101825e+07,349279.179225,130979.692209
2,1.316080e+07,4.935302e+06,0.000000,0.000000,809332.743911,303499.778967,107.199600,40.199850,3.055281e+06,1.145730e+06,...,1830.345443,686.379541,2989.935490,1121.225809,11109.988295,4166.245611,3.036286e+07,1.138607e+07,340369.779635,127638.667363
3,1.289691e+07,4.836340e+06,0.000000,0.000000,795338.555145,298251.958180,70.711454,26.516795,2.790288e+06,1.046358e+06,...,1900.179145,712.567179,0.000000,0.000000,10088.438813,3783.164555,3.146494e+07,1.179935e+07,331257.757247,124221.658968
4,1.196243e+07,4.485911e+06,0.000000,0.000000,781655.961138,293120.985427,26.523142,9.946178,2.540861e+06,9.528230e+05,...,1988.692454,745.759670,0.000000,0.000000,9001.151673,3375.431877,3.261144e+07,1.222929e+07,323151.473668,121181.802625
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28763,0.000000e+00,0.000000e+00,0.000000,0.000000,473511.565561,177566.837086,0.000000,0.000000,0.000000e+00,0.000000e+00,...,3870.007001,1451.252625,0.000000,0.000000,0.000000,0.000000,3.947651e+07,1.480369e+07,99208.904293,37203.339110
28764,0.000000e+00,0.000000e+00,0.000000,0.000000,465558.147739,174584.305402,0.000000,0.000000,0.000000e+00,0.000000e+00,...,3919.413958,1469.780234,0.000000,0.000000,0.000000,0.000000,3.970661e+07,1.488998e+07,92628.244117,34735.591544
28765,0.000000e+00,0.000000e+00,0.000000,0.000000,458111.878346,171791.954380,0.000000,0.000000,0.000000e+00,0.000000e+00,...,3972.905875,1489.839703,0.000000,0.000000,0.000000,0.000000,3.995156e+07,1.498183e+07,86346.664049,32379.999018
28766,0.000000e+00,0.000000e+00,0.000000,0.000000,451145.486631,169179.557487,0.000000,0.000000,0.000000e+00,0.000000e+00,...,4030.638495,1511.489435,0.000000,0.000000,0.000000,0.000000,4.021273e+07,1.507977e+07,80343.859049,30128.947143


In [53]:
# Sum all the subsector emission columns across axis=1
la_production_cost_total_df = ind_fuel_demand_by_sector.copy()
la_production_cost_total_df["production_cost_total"] = la_production_cost_total_df[industry_cost_vars].sum(axis=1)
la_production_cost_total_df.head()

,primary_id,time_period,energy_demand_cement_coal,energy_demand_capex_cement_coal,energy_demand_opex_cement_coal,efficiency_energy_saving_cement_coal,efficiency_capex_cement_coal,efficiency_opex_cement_coal,energy_demand_chemicals_coal,energy_demand_capex_chemicals_coal,...,efficiency_energy_saving_other_product_manufacturing_oil,efficiency_capex_other_product_manufacturing_oil,efficiency_opex_other_product_manufacturing_oil,energy_demand_recycled_wood_oil,energy_demand_capex_recycled_wood_oil,energy_demand_opex_recycled_wood_oil,efficiency_energy_saving_recycled_wood_oil,efficiency_capex_recycled_wood_oil,efficiency_opex_recycled_wood_oil,production_cost_total
0,332332,7,15.110409,1.680276e+07,6.301036e+06,0.000000,0.000000e+00,0.0,0.000732,813.906668,...,0.000000,0.000000e+00,0.0,0.319755,355567.121778,133337.670667,0.000000,0.000000,0.0,4.617921e+08
1,332332,8,12.290454,1.366697e+07,5.125115e+06,4.762860,4.762860e+07,0.0,0.000070,77.506444,...,-1.019247,-1.019247e+07,-0.0,0.314100,349279.179225,130979.692209,0.009181,91811.235443,0.0,4.568581e+08
2,332332,9,11.835264,1.316080e+07,4.935302e+06,5.593867,5.593867e+07,0.0,0.000000,0.000000,...,-2.041192,-2.041192e+07,-0.0,0.306088,340369.779635,127638.667363,0.021401,214010.648444,0.0,4.525865e+08
3,332332,10,11.597945,1.289691e+07,4.836340e+06,6.061209,6.061209e+07,0.0,0.000000,0.000000,...,-3.197407,-3.197407e+07,-0.0,0.297894,331257.757247,124221.658968,0.033769,337688.056982,0.0,4.492909e+08
4,332332,11,10.757587,1.196243e+07,4.485911e+06,7.510201,7.510201e+07,0.0,0.000000,0.000000,...,-4.395974,-4.395974e+07,-0.0,0.290604,323151.473668,121181.802625,0.044862,448619.347968,0.0,4.464197e+08


In [54]:
# Keep only the primary_id, time_period, and emission_total columns
la_production_cost_total_df = la_production_cost_total_df[["primary_id", "time_period", "production_cost_total"]]
la_production_cost_total_df.head()

,primary_id,time_period,production_cost_total
0,332332,7,4.617921e+08
1,332332,8,4.568581e+08
2,332332,9,4.525865e+08
3,332332,10,4.492909e+08
4,332332,11,4.464197e+08


In [55]:
# aggregate data by primary_id summing the emissions
la_production_cost_df_sum_agg = la_production_cost_total_df.groupby(["primary_id"]).sum().reset_index()

# Drop time period column
la_production_cost_df_sum_agg = la_production_cost_df_sum_agg.drop(columns=["time_period"])
la_production_cost_df_sum_agg.head()

,primary_id,production_cost_total
0,332332,1.316782e+10
1,402402,1.306324e+10
2,402403,1.209662e+10
3,402404,1.427672e+10
4,402405,1.501253e+10


In [59]:
la_production_df_sum_agg.shape

(992, 2)

### CB data

In [56]:
# aggregate data by primary_id and region by summing the technical cost
cb_df_agg = cb_df.groupby(["primary_id", "future_id"]).sum().reset_index()
cb_df_agg

,primary_id,future_id,year,technical_cost,consumer_savings,air_pollution
0,402402,0,59044.0,36.079019,12.309442,58.101415
1,402403,1,59044.0,72.057617,10.909213,39.031999
2,402404,2,59044.0,-13.958464,9.862279,20.164441
3,402405,3,59044.0,35.944249,9.606087,35.268119
4,402406,4,59044.0,-5.660414,9.917888,34.670174
...,...,...,...,...,...,...
986,403398,996,59044.0,-93.437962,12.893414,26.609582
987,403399,997,59044.0,-13.462689,9.414455,30.589680
988,403400,998,59044.0,15.617423,10.210951,31.434198
989,403401,999,59044.0,-23.834122,12.109699,26.124292


In [57]:
# Drop year column as it is no longer needed
cb_df_agg = cb_df_agg.drop(columns=["year"], errors='ignore')
cb_df_agg.head()

,primary_id,future_id,technical_cost,consumer_savings,air_pollution
0,402402,0,36.079019,12.309442,58.101415
1,402403,1,72.057617,10.909213,39.031999
2,402404,2,-13.958464,9.862279,20.164441
3,402405,3,35.944249,9.606087,35.268119
4,402406,4,-5.660414,9.917888,34.670174


In [58]:
cb_df_agg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 991 entries, 0 to 990
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   primary_id        991 non-null    int64  
 1   future_id         991 non-null    int64  
 2   technical_cost    991 non-null    float64
 3   consumer_savings  991 non-null    float64
 4   air_pollution     991 non-null    float64
dtypes: float64(3), int64(2)
memory usage: 38.8 KB


## Merge emissions and cb data with lhs samples

In [59]:
attr_primary_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "ATTRIBUTE_PRIMARY_6004_filtered_metamodel_version.csv"))
attr_primary_df

,primary_id,design_id,strategy_id,future_id
0,402402,4,6004,0
1,402403,4,6004,1
2,402404,4,6004,2
3,402405,4,6004,3
4,402406,4,6004,4
...,...,...,...,...
986,403398,4,6004,996
987,403399,4,6004,997
988,403400,4,6004,998
989,403401,4,6004,999


In [60]:
# Check for duplicates in primary_id
duplicates_primary = attr_primary_df[attr_primary_df.duplicated(subset=["primary_id"], keep=False)]
if not duplicates_primary.empty:
    print("Duplicated primary_id found:")
    print(duplicates_primary)
else:
    print("No duplicated primary_id found.")

# Check for duplicates in future_id
duplicates_future = attr_primary_df[attr_primary_df.duplicated(subset=["future_id"], keep=False)]
if not duplicates_future.empty:
    print("Duplicated future_id found:")
    print(duplicates_future)
else:
    print("No duplicated future_id found.")


No duplicated primary_id found.
No duplicated future_id found.


In [62]:
la_emission_df_w_future_id = la_emissions_df_merged.merge(attr_primary_df, on="primary_id", how="inner")

# Drop design_id and stratgy_id columns
la_emission_df_w_future_id = la_emission_df_w_future_id.drop(columns=["design_id", "strategy_id"])
la_emission_df_w_future_id

,primary_id,emission_avg_last_five_years,emission_total,future_id
0,402402,63.023067,4717.592913,0
1,402403,198.512688,6648.349151,1
2,402404,228.001674,7129.331813,2
3,402405,182.853198,6396.992675,3
4,402406,132.272042,5771.905648,4
...,...,...,...,...
986,403398,241.127083,7358.277860,996
987,403399,237.380736,7299.360163,997
988,403400,146.886074,5917.675680,998
989,403401,155.859387,6063.294955,999


In [63]:
la_ssp_out_df = la_emission_df_w_future_id.merge(la_production_df_sum_agg, on="primary_id", how="inner")
la_ssp_out_df = la_ssp_out_df.merge(la_production_cost_df_sum_agg, on="primary_id", how="inner")
la_ssp_out_df.head()

,primary_id,emission_avg_last_five_years,emission_total,future_id,production_total,production_cost_total
0,402402,63.023067,4717.592913,0,6.787144e+08,1.306324e+10
1,402403,198.512688,6648.349151,1,9.944817e+08,1.209662e+10
2,402404,228.001674,7129.331813,2,7.733990e+08,1.427672e+10
3,402405,182.853198,6396.992675,3,7.860210e+08,1.501253e+10
4,402406,132.272042,5771.905648,4,7.332484e+08,1.419782e+10


In [64]:
# Check that the shape is correct
print("la_ssp_out_df shape:", la_ssp_out_df.shape)
print("la_emission_df_w_future_id shape:", la_emission_df_w_future_id.shape)
print("la_production_df_sum_agg shape:", la_production_df_sum_agg.shape)
print("la_production_cost_df_sum_agg shape:", la_production_cost_df_sum_agg.shape)

la_ssp_out_df shape: (991, 6)
la_emission_df_w_future_id shape: (991, 4)
la_production_df_sum_agg shape: (992, 2)
la_production_cost_df_sum_agg shape: (992, 2)


In [65]:
la_ssp_out_df.isna().sum()

primary_id                      0
emission_avg_last_five_years    0
emission_total                  0
future_id                       0
production_total                0
production_cost_total           0
dtype: int64

In [66]:
lhs_df_merged.head()

,future_id,47,48,49,50,51,52,53,54,55,...,1753,1754,1759,1760,1762,1770,1772,1784,1785,1793
1000,1,0.639363,0.261333,0.115895,0.456909,0.389376,0.065254,0.452244,0.395609,0.190891,...,0.406492,0.785543,0.249585,0.808708,0.824054,0.596933,0.085905,0.349450,0.718493,0.920739
1001,2,0.524422,0.141191,0.625291,0.369740,0.475667,0.017979,0.912414,0.894009,0.046745,...,0.133208,0.725231,0.615082,0.234993,0.342072,0.975031,0.431114,0.700254,0.246668,0.536234
1002,3,0.438775,0.902956,0.099015,0.511602,0.405292,0.416929,0.832413,0.406103,0.929365,...,0.320059,0.836571,0.386288,0.728487,0.069142,0.128675,0.699301,0.166701,0.354644,0.300748
1003,4,0.025503,0.833553,0.666840,0.301819,0.569698,0.778987,0.995629,0.994153,0.946893,...,0.318834,0.118582,0.767219,0.762363,0.120502,0.838756,0.298101,0.885792,0.274482,0.827713
1004,5,0.225323,0.718339,0.291168,0.507558,0.499419,0.918567,0.183357,0.397512,0.328322,...,0.954028,0.331731,0.708760,0.200289,0.889109,0.238305,0.473463,0.147280,0.491020,0.480479


In [67]:
lhs_emissions_merged_df = pd.merge(lhs_df_merged, la_ssp_out_df, on="future_id", how="inner")
lhs_emissions_merged_df.head()

,future_id,47,48,49,50,51,52,53,54,55,...,1770,1772,1784,1785,1793,primary_id,emission_avg_last_five_years,emission_total,production_total,production_cost_total
0,1,0.639363,0.261333,0.115895,0.456909,0.389376,0.065254,0.452244,0.395609,0.190891,...,0.596933,0.085905,0.349450,0.718493,0.920739,402403,198.512688,6648.349151,9.944817e+08,1.209662e+10
1,2,0.524422,0.141191,0.625291,0.369740,0.475667,0.017979,0.912414,0.894009,0.046745,...,0.975031,0.431114,0.700254,0.246668,0.536234,402404,228.001674,7129.331813,7.733990e+08,1.427672e+10
2,3,0.438775,0.902956,0.099015,0.511602,0.405292,0.416929,0.832413,0.406103,0.929365,...,0.128675,0.699301,0.166701,0.354644,0.300748,402405,182.853198,6396.992675,7.860210e+08,1.501253e+10
3,4,0.025503,0.833553,0.666840,0.301819,0.569698,0.778987,0.995629,0.994153,0.946893,...,0.838756,0.298101,0.885792,0.274482,0.827713,402406,132.272042,5771.905648,7.332484e+08,1.419782e+10
4,5,0.225323,0.718339,0.291168,0.507558,0.499419,0.918567,0.183357,0.397512,0.328322,...,0.238305,0.473463,0.147280,0.491020,0.480479,402407,173.059694,6393.870401,9.001498e+08,1.355211e+10


In [68]:
lhs_emissions_merged_df.shape

(990, 88)

In [69]:
cb_df_agg.head()

,primary_id,future_id,technical_cost,consumer_savings,air_pollution
0,402402,0,36.079019,12.309442,58.101415
1,402403,1,72.057617,10.909213,39.031999
2,402404,2,-13.958464,9.862279,20.164441
3,402405,3,35.944249,9.606087,35.268119
4,402406,4,-5.660414,9.917888,34.670174


In [70]:
complete_merged_df = pd.merge(lhs_emissions_merged_df, cb_df_agg, on=["future_id", "primary_id"], how="inner")
complete_merged_df.head()

,future_id,47,48,49,50,51,52,53,54,55,...,1785,1793,primary_id,emission_avg_last_five_years,emission_total,production_total,production_cost_total,technical_cost,consumer_savings,air_pollution
0,1,0.639363,0.261333,0.115895,0.456909,0.389376,0.065254,0.452244,0.395609,0.190891,...,0.718493,0.920739,402403,198.512688,6648.349151,9.944817e+08,1.209662e+10,72.057617,10.909213,39.031999
1,2,0.524422,0.141191,0.625291,0.369740,0.475667,0.017979,0.912414,0.894009,0.046745,...,0.246668,0.536234,402404,228.001674,7129.331813,7.733990e+08,1.427672e+10,-13.958464,9.862279,20.164441
2,3,0.438775,0.902956,0.099015,0.511602,0.405292,0.416929,0.832413,0.406103,0.929365,...,0.354644,0.300748,402405,182.853198,6396.992675,7.860210e+08,1.501253e+10,35.944249,9.606087,35.268119
3,4,0.025503,0.833553,0.666840,0.301819,0.569698,0.778987,0.995629,0.994153,0.946893,...,0.274482,0.827713,402406,132.272042,5771.905648,7.332484e+08,1.419782e+10,-5.660414,9.917888,34.670174
4,5,0.225323,0.718339,0.291168,0.507558,0.499419,0.918567,0.183357,0.397512,0.328322,...,0.491020,0.480479,402407,173.059694,6393.870401,9.001498e+08,1.355211e+10,-14.126208,9.132959,32.423383


In [71]:
print(complete_merged_df.shape)
print(complete_merged_df.future_id.nunique())

(990, 91)
990


In [72]:
complete_merged_df.isna().sum().sum()

np.int64(0)

In [73]:
# rearrange columns to have future_id and primary_id at the front
cols_order = ["future_id", "primary_id"] + [col for col in complete_merged_df.columns if col not in ["future_id", "primary_id"]]
complete_merged_df = complete_merged_df[cols_order]
complete_merged_df

,future_id,primary_id,47,48,49,50,51,52,53,54,...,1784,1785,1793,emission_avg_last_five_years,emission_total,production_total,production_cost_total,technical_cost,consumer_savings,air_pollution
0,1,402403,0.639363,0.261333,0.115895,0.456909,0.389376,0.065254,0.452244,0.395609,...,0.349450,0.718493,0.920739,198.512688,6648.349151,9.944817e+08,1.209662e+10,72.057617,10.909213,39.031999
1,2,402404,0.524422,0.141191,0.625291,0.369740,0.475667,0.017979,0.912414,0.894009,...,0.700254,0.246668,0.536234,228.001674,7129.331813,7.733990e+08,1.427672e+10,-13.958464,9.862279,20.164441
2,3,402405,0.438775,0.902956,0.099015,0.511602,0.405292,0.416929,0.832413,0.406103,...,0.166701,0.354644,0.300748,182.853198,6396.992675,7.860210e+08,1.501253e+10,35.944249,9.606087,35.268119
3,4,402406,0.025503,0.833553,0.666840,0.301819,0.569698,0.778987,0.995629,0.994153,...,0.885792,0.274482,0.827713,132.272042,5771.905648,7.332484e+08,1.419782e+10,-5.660414,9.917888,34.670174
4,5,402407,0.225323,0.718339,0.291168,0.507558,0.499419,0.918567,0.183357,0.397512,...,0.147280,0.491020,0.480479,173.059694,6393.870401,9.001498e+08,1.355211e+10,-14.126208,9.132959,32.423383
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
985,996,403398,0.736701,0.546430,0.739303,0.054172,0.451181,0.875108,0.854104,0.601669,...,0.710591,0.289082,0.437041,241.127083,7358.277860,1.029956e+09,1.267728e+10,-93.437962,12.893414,26.609582
986,997,403399,0.484244,0.232193,0.411141,0.677399,0.536223,0.275901,0.268436,0.277561,...,0.496825,0.394089,0.772607,237.380736,7299.360163,1.149071e+09,1.313187e+10,-13.462689,9.414455,30.589680
987,998,403400,0.594076,0.453653,0.999049,0.260340,0.276599,0.669046,0.335968,0.251319,...,0.916280,0.551641,0.511806,146.886074,5917.675680,7.394170e+08,1.371479e+10,15.617423,10.210951,31.434198
988,999,403401,0.991327,0.185911,0.093308,0.002969,0.027930,0.843634,0.806122,0.438407,...,0.489433,0.386625,0.828784,155.859387,6063.294955,7.567117e+08,1.316515e+10,-23.834122,12.109699,26.124292


In [74]:
# Check for nans
complete_merged_df.isna().sum().sum()

np.int64(0)

## Filter out irrelevant lhs groups

In [75]:
var_traj_X_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "VARIABLE_TRAJECTORY_GROUPS_X.csv"))
var_traj_L_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "VARIABLE_TRAJECTORY_GROUPS_L.csv"))

In [76]:
var_traj_X_df.tail()

,variable,variable_trajectory_group
66,elasticity_ippu_wood_production_to_gdp,57
67,elasticity_ippu_product_use_lubricants_product...,58
68,elasticity_ippu_product_use_ods_other_product_...,58
69,elasticity_ippu_product_use_ods_refrigeration_...,58
70,elasticity_ippu_product_use_paraffin_wax_produ...,58


In [77]:
var_traj_L_df.tail()

,transformation_code,variable,variable_field,variable_trajectory_group
462,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_paper,46
463,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_plastic,46
464,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_rubber_leather,46
465,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_textiles,46
466,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_wood,46


In [78]:
var_traj_groups_X = var_traj_X_df["variable_trajectory_group"].unique()
var_traj_groups_L = var_traj_L_df["variable_trajectory_group"].unique()
print("Variable trajectory groups X:", var_traj_groups_X)
print("Variable trajectory groups L:", var_traj_groups_L)

Variable trajectory groups X: [47 48 49 50 51 52 53 54 55 56 57 58]
Variable trajectory groups L: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46]


In [79]:
# join the var_traj_groups
var_traj_groups_all = var_traj_groups_X.tolist() + var_traj_groups_L.tolist()
var_traj_groups_all = list(set(var_traj_groups_all))  # remove duplicates
print("All variable trajectory groups:", var_traj_groups_all)

All variable trajectory groups: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58]


In [80]:
# Convert the variable_trajectory_group column to list of strings
relevant_lhs_cols = [str(col) for col in var_traj_groups_all]

In [81]:
df_cols = complete_merged_df.columns.tolist()

# Filter the relevant_lhs_cols to only include those that are in df_cols
relevant_lhs_cols = [col for col in relevant_lhs_cols if col in df_cols]

In [82]:
# filter complete_merged_df to keep only relevant columns
cols_to_keep = ["future_id", "primary_id"] + list(relevant_lhs_cols) + ["emission_avg_last_five_years", "emission_total", "production_total","production_cost_total","technical_cost", "air_pollution" ,"consumer_savings"]
merged_df_filtered = complete_merged_df[cols_to_keep]

In [83]:
print("Original merged DataFrame shape:", complete_merged_df.shape)
print("Filtered merged DataFrame shape:", merged_df_filtered.shape)

Original merged DataFrame shape: (990, 91)
Filtered merged DataFrame shape: (990, 67)


In [84]:
print("Filtered merged DataFrame fields:", merged_df_filtered.columns.tolist())
print("Relevant LHS columns:", relevant_lhs_cols)

Filtered merged DataFrame fields: ['future_id', 'primary_id', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', 'emission_avg_last_five_years', 'emission_total', 'production_total', 'production_cost_total', 'technical_cost', 'air_pollution', 'consumer_savings']
Relevant LHS columns: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58']


In [85]:
merged_df_filtered.head()

,future_id,primary_id,1,2,3,4,5,6,7,8,...,56,57,58,emission_avg_last_five_years,emission_total,production_total,production_cost_total,technical_cost,air_pollution,consumer_savings
0,1,402403,0.354682,0.681300,0.670551,0.371381,0.597378,0.390717,0.554668,0.176302,...,0.681412,0.199758,0.969220,198.512688,6648.349151,9.944817e+08,1.209662e+10,72.057617,39.031999,10.909213
1,2,402404,0.023662,0.659807,0.066549,0.350365,0.536160,0.040042,0.101776,0.683933,...,0.000798,0.725034,0.902345,228.001674,7129.331813,7.733990e+08,1.427672e+10,-13.958464,20.164441,9.862279
2,3,402405,0.414641,0.092022,0.227137,0.020454,0.832906,0.293800,0.601890,0.263913,...,0.575067,0.208350,0.175365,182.853198,6396.992675,7.860210e+08,1.501253e+10,35.944249,35.268119,9.606087
3,4,402406,0.307337,0.204967,0.050181,0.805452,0.558369,0.506111,0.974177,0.923617,...,0.506764,0.227383,0.113647,132.272042,5771.905648,7.332484e+08,1.419782e+10,-5.660414,34.670174,9.917888
4,5,402407,0.800355,0.458285,0.101171,0.738594,0.254548,0.034029,0.673997,0.851671,...,0.362234,0.947292,0.764492,173.059694,6393.870401,9.001498e+08,1.355211e+10,-14.126208,32.423383,9.132959


## Add variable names to lhs columns

In [86]:
var_traj_L_df.tail()

,transformation_code,variable,variable_field,variable_trajectory_group
462,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_paper,46
463,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_plastic,46
464,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_rubber_leather,46
465,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_textiles,46
466,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_wood,46


In [87]:
var_traj_L_df = var_traj_L_df[["variable_field", "variable_trajectory_group"]]
var_traj_L_df = var_traj_L_df.rename(columns={"variable_field": "variable"})
var_traj_L_df.head()

,variable,variable_trajectory_group
0,ef_agrc_anaerobicdom_rice_kg_ch4_ha,1
1,frac_agrc_agriculture_production_lost,2
2,frac_agrc_crop_residues_burned,3
3,frac_agrc_crop_residues_removed,3
4,frac_agrc_no_till_cereals,3


In [88]:
var_traj_X_df.tail()

,variable,variable_trajectory_group
66,elasticity_ippu_wood_production_to_gdp,57
67,elasticity_ippu_product_use_lubricants_product...,58
68,elasticity_ippu_product_use_ods_other_product_...,58
69,elasticity_ippu_product_use_ods_refrigeration_...,58
70,elasticity_ippu_product_use_paraffin_wax_produ...,58


In [89]:
var_traj_all_df = pd.concat([var_traj_X_df, var_traj_L_df], ignore_index=True)
var_traj_all_df

,variable,variable_trajectory_group
0,cost_enfu_fuel_coal_usd_per_tonne,47
1,cost_enfu_fuel_coke_usd_per_tonne,47
2,cost_enfu_fuel_hydrocarbon_gas_liquids_usd_per...,47
3,cost_enfu_fuel_natural_gas_usd_per_mmbtu,47
4,cost_enfu_fuel_crude_usd_per_m3,47
...,...,...
533,frac_waso_recycled_paper,46
534,frac_waso_recycled_plastic,46
535,frac_waso_recycled_rubber_leather,46
536,frac_waso_recycled_textiles,46


In [90]:
# drop duplicates if any
print("Before dropping duplicates, var_traj_all_df shape:", var_traj_all_df.shape)
var_traj_all_df = var_traj_all_df.drop_duplicates(subset=["variable", "variable_trajectory_group"])
print("After dropping duplicates, var_traj_all_df shape:", var_traj_all_df.shape)

Before dropping duplicates, var_traj_all_df shape: (538, 2)
After dropping duplicates, var_traj_all_df shape: (530, 2)


In [91]:
# check if there are any duplicated variable names
duplicated_vars = var_traj_all_df["variable"].duplicated().any()
if duplicated_vars:
    print("There are duplicated variable names in var_traj_all_df.")
else:
    print("No duplicated variable names in var_traj_all_df.")

No duplicated variable names in var_traj_all_df.


In [92]:
# Filter var_traj_all_df by sample_group in relevant_lhs_cols
relevant_lhs_cols = [int(col) for col in relevant_lhs_cols]
var_traj_all_df = var_traj_all_df[var_traj_all_df["variable_trajectory_group"].isin(relevant_lhs_cols)]
var_traj_all_df = var_traj_all_df.sort_values(by="variable_trajectory_group", ascending=True)
print("After filtering by relevant_lhs_cols, var_traj_all_df shape:", var_traj_all_df.shape)

After filtering by relevant_lhs_cols, var_traj_all_df shape: (530, 2)


In [94]:
def process_variable_prefix(df):
    result = []
    for group, group_df in df.groupby('variable_trajectory_group'):
        variables = group_df['variable'].tolist()
        if len(variables) == 1:
            prefix = variables[0]
        else:
            prefix = os.path.commonprefix(variables)
            # Clean trailing underscores
            prefix = prefix.rstrip('_')
            
        prefix = f"group_{group}_{prefix}"
        result.append({'variable_trajectory_group': group, 'variable_prefix': prefix})
    return pd.DataFrame(result)

prefix_df = process_variable_prefix(var_traj_all_df)
prefix_df

,variable_trajectory_group,variable_prefix
0,1,group_1_ef_agrc_anaerobicdom_rice_kg_ch4_ha
1,2,group_2_frac_agrc_agriculture_production_lost
2,3,group_3_frac_agrc
3,4,group_4_qty_ccsq_mt_co2_captured_sequestered_b...
4,5,group_5_frac_enfu_transmission_loss_fuel_elect...
5,6,group_6_nemomod_entc_frac_min_share_production...
6,7,group_7_nemomod_en
7,8,group_8_frac_fgtv_reduction_in_fugitive_leaks
8,9,group_9_frac_fgtv_drained_and_waste_ch4_flared...
9,10,group_10_efficfactor_enfu_industrial_energy_fuel


In [95]:
# Check for duplicates in variable_trajectory_group and variable_prefix
dups = prefix_df.duplicated(subset=["variable_trajectory_group", "variable_prefix"], keep=False)
if dups.any():
    print("Duplicated variable_trajectory_group and variable_prefix found:")
    print(prefix_df[dups])
else:
    print("No duplicated variable_trajectory_group and variable_prefix found.")

# Check for duplicates in variable_trajectory_group
dups_group = prefix_df.duplicated(subset=["variable_trajectory_group"], keep=False)
if dups_group.any():
    print("Duplicated variable_trajectory_group found:")
    print(prefix_df[dups_group])
else:
    print("No duplicated variable_trajectory_group found.")

# Check for duplicates in variable_prefix
dups_prefix = prefix_df.duplicated(subset=["variable_prefix"], keep=False)
if dups_prefix.any():
    print("Duplicated variable_prefix found:")
    print(prefix_df[dups_prefix])
else:
    print("No duplicated variable_prefix found.")

No duplicated variable_trajectory_group and variable_prefix found.
No duplicated variable_trajectory_group found.
No duplicated variable_prefix found.


In [104]:
# var_traj_all_df[var_traj_all_df["variable_trajectory_group"].isin([3, 13, 40])]

In [105]:
# prefix_df.loc[prefix_df["sample_group"] == 13, "variable_prefix"] = "group_13_frac_gnrl_eating_red_meats+"
# prefix_df.loc[prefix_df["sample_group"] == 40, "variable_prefix"] = "group_40_pij_lndu_grasslands+"

# prefix_df = prefix_df.sort_values(by="variable_prefix", ascending=True)
# prefix_df

In [96]:
# Let's use the prefix_df to rename the columns in merged_df_filtered
def rename_columns_with_prefix(merged_df, prefix_df):
    df = merged_df.copy()
    # Create a mapping from str(group) to prefix
    group_to_prefix = {str(row['variable_trajectory_group']): row['variable_prefix'] for _, row in prefix_df.iterrows()}
    # Only rename columns that match a group
    rename_dict = {col: group_to_prefix[col] for col in df.columns if col in group_to_prefix}
    df = df.rename(columns=rename_dict)
    return df

merged_df_filtered_w_prefix = rename_columns_with_prefix(merged_df_filtered, prefix_df)

In [97]:
merged_df_filtered_w_prefix

,future_id,primary_id,group_1_ef_agrc_anaerobicdom_rice_kg_ch4_ha,group_2_frac_agrc_agriculture_production_lost,group_3_frac_agrc,group_4_qty_ccsq_mt_co2_captured_sequestered_by_direct_air_capture,group_5_frac_enfu_transmission_loss_fuel_electricity,group_6_nemomod_entc_frac_min_share_production_fp_hydrogen_electrolysis,group_7_nemomod_en,group_8_frac_fgtv_reduction_in_fugitive_leaks,...,group_56_elasticity_trde_pkm_to_gdppc,group_57_elasticity_ippu,group_58_elasticity_ippu_product_use,emission_avg_last_five_years,emission_total,production_total,production_cost_total,technical_cost,air_pollution,consumer_savings
0,1,402403,0.354682,0.681300,0.670551,0.371381,0.597378,0.390717,0.554668,0.176302,...,0.681412,0.199758,0.969220,198.512688,6648.349151,9.944817e+08,1.209662e+10,72.057617,39.031999,10.909213
1,2,402404,0.023662,0.659807,0.066549,0.350365,0.536160,0.040042,0.101776,0.683933,...,0.000798,0.725034,0.902345,228.001674,7129.331813,7.733990e+08,1.427672e+10,-13.958464,20.164441,9.862279
2,3,402405,0.414641,0.092022,0.227137,0.020454,0.832906,0.293800,0.601890,0.263913,...,0.575067,0.208350,0.175365,182.853198,6396.992675,7.860210e+08,1.501253e+10,35.944249,35.268119,9.606087
3,4,402406,0.307337,0.204967,0.050181,0.805452,0.558369,0.506111,0.974177,0.923617,...,0.506764,0.227383,0.113647,132.272042,5771.905648,7.332484e+08,1.419782e+10,-5.660414,34.670174,9.917888
4,5,402407,0.800355,0.458285,0.101171,0.738594,0.254548,0.034029,0.673997,0.851671,...,0.362234,0.947292,0.764492,173.059694,6393.870401,9.001498e+08,1.355211e+10,-14.126208,32.423383,9.132959
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
985,996,403398,0.164099,0.545603,0.990562,0.405820,0.629065,0.929753,0.199855,0.816744,...,0.879831,0.204425,0.508420,241.127083,7358.277860,1.029956e+09,1.267728e+10,-93.437962,26.609582,12.893414
986,997,403399,0.873192,0.273625,0.486877,0.773743,0.644259,0.566471,0.152724,0.244781,...,0.579293,0.826772,0.510035,237.380736,7299.360163,1.149071e+09,1.313187e+10,-13.462689,30.589680,9.414455
987,998,403400,0.465877,0.906249,0.406608,0.797448,0.516324,0.264986,0.765120,0.227728,...,0.278802,0.359322,0.344483,146.886074,5917.675680,7.394170e+08,1.371479e+10,15.617423,31.434198,10.210951
988,999,403401,0.128720,0.865683,0.939286,0.982105,0.130143,0.036545,0.276138,0.666679,...,0.222832,0.415388,0.434470,155.859387,6063.294955,7.567117e+08,1.316515e+10,-23.834122,26.124292,12.109699


In [98]:
print(merged_df_filtered.shape)
print(merged_df_filtered_w_prefix.shape)

(990, 67)
(990, 67)


In [99]:
# check for duplicated column names
duplicated_cols = merged_df_filtered_w_prefix.columns[merged_df_filtered_w_prefix.columns.duplicated()].tolist()
if duplicated_cols:
    print("Duplicated column names found:", duplicated_cols)
else:
    print("No duplicated column names found.")

No duplicated column names found.


## Finally we save the processed data as training data

In [100]:
#save the merged DataFrame to a CSV file
merged_df_filtered_w_prefix.to_csv(os.path.join(TRAINING_DIR_PATH, "training_data_v5.3_without_health.csv"), index=False)